# CS 445 Final Project Spring 2025

Day-to-night image converter by Mayank Hirani (mhirani2) and Nikitha Sundaram (nikitha7)

# Notes

Two current ideas of approaches

1. ControlNet with some sort of stable diffusion or image generation to fill in the image, then use color modification methods like the ones from MP1 (or MP3?) to shift the balance of colors to match the original image

Running out of RAM currently while running. Maybe switch to using simpler generation model like GAN or do second approach?

https://github.com/lllyasviel/ControlNet


2. Use pix2pixHD and improve on it. Try training on noisy and modified images to improve pix2pixHD accuracy, similar to how they did so to improve signal processing in this paper:

https://github.com/SamsungLabs/day-to-night/tree/main

https://openaccess.thecvf.com/content/CVPR2022/papers/Punnappurath_Day-to-Night_Image_Synthesis_for_Training_Nighttime_Neural_ISPs_CVPR_2022_paper.pdf


Datasets

https://github.com/isurushanaka/paired-N2D (Dataset used)

https://www.kaggle.com/datasets/stevemark/daynight-dataset?resource=download

https://soonminhwang.github.io/rgbt-ped-detection/



# Imports

In [1]:
from google.colab import drive
drive.mount('/content/drive')

datadir = "/content/drive/My Drive/CS 445 Final Project/"

first_time_running = False

%cd "{datadir}"

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1srzIn3tl3HzMkMxEitOGOH4n2Z9NtubB/CS 445 Final Project


In [2]:
# !pip install diffusers transformers accelerate torch opencv-python xformers
!pip install dominate
# !pip install xformers

In [ ]:
# To fix torchvision conflicting versions with cuda

# !pip uninstall -y torch torchvision
# !pip cache purge
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# Import model (only needed first time if copied to drive)

# !git clone https://github.com/NVIDIA/pix2pixHD.git
# !cd pix2pixHD/
# !pip install -r requirements.txt

Cloning into 'pix2pixHD'...
remote: Enumerating objects: 343, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 343 (delta 0), reused 0 (delta 0), pack-reused 340 (from 1)
Receiving objects: 100% (343/343), 55.68 MiB | 6.71 MiB/s, done.
Resolving deltas: 100% (156/156), done.


In [ ]:
# Only run first time, for copying pix2pixHD source code in

# !cp -r pix2pixHD/ /content/drive/MyDrive/CS\ 445\ Final\ Project/pix2pixHD

# Prepare Input Data

In [ ]:
# Separating combined images into two directories train_A and train_B using left side as night and right as day
# ONLY RUN FIRST TIME

# import os
# from PIL import Image

# # Swap day and night

# # # Input folder with side-by-side images (night | day)
# # input_dir = datadir + "data/dataset_input"
# # # Output folders
# # output_dir = datadir + "data/dataset"

# # # Process each image
# # for fname in os.listdir(input_dir):
# #     if not fname.lower().endswith(('.jpg', '.png', '.jpeg')):
# #         continue

# #     print(fname)

# #     img_path = os.path.join(input_dir, fname)
# #     img = Image.open(img_path).convert("RGB")

# #     # Split left (night) and right (day)
# #     width, height = img.size

# #     night = img.crop((0, 0, 256, 256))   # (left, top, right, bottom)
# #     day = img.crop((256, 0, 512, 256))

# #     # Concatenate as day (input) → night (target), as expected by pix2pix
# #     combined = Image.new("RGB", (512, 256))
# #     combined.paste(day, (0, 0))   # input on the left
# #     combined.paste(night, (256, 0))  # target on the right

# #     # Save the training pair
# #     combined.save(os.path.join(output_dir, fname))

# # Seperate into A and B folders (ran after swapping, run separately)

# input_dir = datadir + "data/dataset/train"  # your original side-by-side images
# output_A = datadir + "data/dataset/train_A"  # day images (input)
# output_B = datadir + "data/dataset/train_B"  # night images (target)

# for i, fname in enumerate(os.listdir(input_dir)):
#     if not fname.lower().endswith(('.jpg', '.png', '.jpeg')):
#         continue

#     img_path = os.path.join(input_dir, fname)
#     img = Image.open(img_path).convert("RGB")
#     w, h = img.size
#     assert w == 512 and h == 256, f"{fname} has wrong size"

#     # [left=day, right=night] — flip if needed
#     day = img.crop((0, 0, 256, 256))
#     night = img.crop((256, 0, 512, 256))

#     day.save(os.path.join(output_A, "img" + str(i) + ".jpg"))
#     night.save(os.path.join(output_B, "img" + str(i) + ".jpg"))

# Add methods from paper onto training input day images

In [ ]:
# First copy all the target night images to the new folder
# ONLY RUN FIRST TIME

# import os
# from PIL import Image
# import numpy as np
# import cv2

# train_original = datadir + "data/dataset/train_B/"
# train_new = datadir + "data/dataset_noise/train_B/"

# # Loop through all training night images and copy them to the noisy dataset folder
# for fname in os.listdir(train_original):
#     img = cv2.imread(train_original + fname)
#     cv2.imwrite(train_new + fname, img)

In [ ]:
# Add methods from paper to make the day images noisy
# ONLY RUN ONCE

import os
from PIL import Image
import numpy as np
import cv2
import matplotlib.pyplot as plt

if first_time_running:

    train_original = datadir + "data/dataset/train_A/"
    train_new = datadir + "data/dataset_noise/train_A/"

    # Loop through all training day images
    for fname in os.listdir(train_original):
        # Load image
        # print(fname)
        day_img = cv2.imread(train_original + fname)

        # plt.imshow(cv2.cvtColor(day_img, cv2.COLOR_BGR2RGB))
        # plt.show()

        day_img = day_img.astype("float32")

        # Apply the steps in the pipeline used in the paper:
        result_img = np.copy(day_img)

        # 1. Normalize
        result_img /= 255.0

        # 2. Remove the day lighting with gamma correction
        result_img = np.power(result_img, 1.5)

        # 3. Model lower exposure
        result_img *= 0.5
        result_img = np.clip(result_img, 0, 1)

        # 4: Add night illumination effect (vignette + color tint)
        # Vignette code from ChatGPT
        h, w = result_img.shape[:2]
        Y, X = np.ogrid[:h, :w]
        center = (h / 2, w / 2)
        vignette = 1 - (((X - center[1])**2 + (Y - center[0])**2) / (w*h))
        vignette = vignette[..., np.newaxis]
        result_img *= vignette.astype("float32")  # apply vignette

        # Apply greenish bluish tint
        tint = np.array([0.9, 1.0, 1.2])
        result_img *= tint
        result_img = np.clip(result_img, 0, 1)


        # 5: Add Gaussian noise
        noise_std = 0.1
        noise = np.random.normal(0, noise_std, result_img.shape).astype("float32")
        result_img = np.clip(result_img + noise, 0, 1)

        # 6: Denormalize back to [0, 255] (swapped steps 5 and 6 from paper, it doesnt matter)
        result_img = (result_img * 255).astype("uint8")

        # Write the output image in dataset_noise/train_A/
        cv2.imwrite(train_new + fname, result_img)

        # plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
        # plt.show()


# Train pix2pixHD on regular day images

In [ ]:
# Removing dataset images to make dataset smaller
# ONLY RAN ONCE

# import os
# from PIL import Image
# import re

# if first_time_running:
#     results_dir = data_path = datadir + "data/dataset/train_A"

#     for fname in os.listdir(results_dir):
#         print(fname)
#         path = os.path.join(results_dir, fname)
#         match = re.search(r'(\d+)', fname)
#         if match:
#             number = int(match.group(1))
#             if number > 1000:
#                 os.remove(path)





frame_R_734_fake_1.jpg
frame_L_885_fake_1.jpg
frame_R_358_fake_1.jpg
frame_R_639_fake_1.jpg
frame_R_972_fake_1.jpg
frame_L_887_fake_1.jpg
frame_R_274_fake_1.jpg
frame_R_54_fake_1.jpg
frame_R_739_fake_1.jpg
frame_R_614_fake_1.jpg
frame_R_139_fake_1.jpg
frame_R_971_fake_1.jpg
frame_L_833_fake_1.jpg
frame_R_420_fake_1.jpg
frame_R_390_fake_1.jpg
frame_L_113_fake_1.jpg
frame_R_957_fake_1.jpg
frame_R_604_fake_1.jpg
frame_L_454_fake_1.jpg
frame_L_150_fake_1.jpg
frame_R_962_fake_1.jpg
frame_L_721_fake_1.jpg
frame_R_330_fake_1.jpg
frame_R_583_fake_1.jpg
frame_L_23_fake_1.jpg
frame_L_21_fake_1.jpg
frame_L_451_fake_1.jpg
frame_R_360_fake_1.jpg
frame_L_614_fake_1.jpg
frame_R_608_fake_1.jpg
frame_R_417_fake_1.jpg
frame_R_329_fake_1.jpg
frame_R_69_fake_1.jpg
frame_R_193_fake_1.jpg
frame_L_637_fake_1.jpg
frame_R_991_fake_1.jpg
frame_L_429_fake_1.jpg
frame_L_469_fake_1.jpg
frame_L_854_fake_1.jpg
frame_R_715_fake_1.jpg
frame_L_711_fake_1.jpg
frame_R_907_fake_1.jpg
frame_R_897_fake_1.jpg
frame_L_713_fak

In [ ]:
# Train on the regular day images as day2night

if first_time_running:
    data_path = datadir + "data/dataset"
    # pix2pix_dir = datadir + "pix2pixHD/"
    !cd pix2pixHD/ && python train.py --name day2night \
        --dataroot "{data_path}" \
        --label_nc 0 \
        --no_instance \
        --niter 1 \
        --niter_decay 0 \
        --save_epoch_freq 1 \
        --batchSize 1 \
        --max_dataset_size 1000 # Force it to only use images that weren't deleted
    # Save the model
    !cp -r pix2pixHD/checkpoints/day2night /content/drive/MyDrive/CS\ 445\ Final\ Project/pix2pixHD/checkpoints/day2night

------------ Options -------------
batchSize: 1
beta1: 0.5
checkpoints_dir: ./checkpoints
continue_train: False
data_type: 32
dataroot: /content/drive/My Drive/CS 445 Final Project/data/dataset
debug: False
display_freq: 100
display_winsize: 512
feat_num: 3
fineSize: 512
fp16: False
gpu_ids: [0]
input_nc: 3
instance_feat: False
isTrain: True
label_feat: False
label_nc: 0
lambda_feat: 10.0
loadSize: 1024
load_features: False
load_pretrain: 
local_rank: 0
lr: 0.0002
max_dataset_size: 1000
model: pix2pixHD
nThreads: 2
n_blocks_global: 9
n_blocks_local: 3
n_clusters: 10
n_downsample_E: 4
n_downsample_global: 4
n_layers_D: 3
n_local_enhancers: 1
name: day2night
ndf: 64
nef: 16
netG: global
ngf: 64
niter: 1
niter_decay: 0
niter_fix_global: 0
no_flip: False
no_ganFeat_loss: False
no_html: False
no_instance: True
no_lsgan: False
no_vgg_loss: False
norm: instance
num_D: 2
output_nc: 3
phase: train
pool_size: 0
print_freq: 100
resize_or_crop: scale_width
save_epoch_freq: 1
save_latest_freq: 1000

# Train pix2pixHD on noisy day images

In [4]:
# Train on the noisy day images as noisy2night

# if first_time_running:
data_path = datadir + "data/dataset_noise"
# pix2pix_dir = datadir + "pix2pixHD/"
!cd pix2pixHD/ && python train.py --name noisy2night \
    --dataroot "{data_path}" \
    --label_nc 0 \
    --no_instance \
    --niter 1 \
    --niter_decay 0 \
    --save_epoch_freq 1 \
    --batchSize 1 \
    --max_dataset_size 1000 # Force it to only use images that weren't deleted
# Save the model
!cp -r pix2pixHD/checkpoints/noisy2night /content/drive/MyDrive/CS\ 445\ Final\ Project/pix2pixHD/checkpoints/noisy2night


------------ Options -------------
batchSize: 1
beta1: 0.5
checkpoints_dir: ./checkpoints
continue_train: False
data_type: 32
dataroot: /content/drive/My Drive/CS 445 Final Project/data/dataset_noise
debug: False
display_freq: 100
display_winsize: 512
feat_num: 3
fineSize: 512
fp16: False
gpu_ids: [0]
input_nc: 3
instance_feat: False
isTrain: True
label_feat: False
label_nc: 0
lambda_feat: 10.0
loadSize: 1024
load_features: False
load_pretrain: 
local_rank: 0
lr: 0.0002
max_dataset_size: 1000
model: pix2pixHD
nThreads: 2
n_blocks_global: 9
n_blocks_local: 3
n_clusters: 10
n_downsample_E: 4
n_downsample_global: 4
n_layers_D: 3
n_local_enhancers: 1
name: noisy2night
ndf: 64
nef: 16
netG: global
ngf: 64
niter: 1
niter_decay: 0
niter_fix_global: 0
no_flip: False
no_ganFeat_loss: False
no_html: False
no_instance: True
no_lsgan: False
no_vgg_loss: False
norm: instance
num_D: 2
output_nc: 3
phase: train
pool_size: 0
print_freq: 100
resize_or_crop: scale_width
save_epoch_freq: 1
save_latest_fr

# Resize all test input images to 256x256 and copy to both test directories

In [5]:
# Copy test images to dataset_noisy too

import os
from PIL import Image
import numpy as np
import cv2

# COPYING

test_original = datadir + "data/dataset/test_A/"
test_new = datadir + "data/dataset_noise/test_A/"

# Loop through all test day images and copy them to the noisy dataset folder
for fname in os.listdir(test_original):
    img = cv2.imread(test_original + fname)
    cv2.imwrite(test_new + fname, img)



# RESIZING

results_dir = datadir + "data/dataset/test_A"

# Resize all images to 256x256 before testing on them
for fname in os.listdir(results_dir):
    # print(fname)
    path = os.path.join(results_dir, fname)

    img = Image.open(path).convert("RGB").resize((256, 256))

    img.save(path)

results_dir = datadir + "data/dataset_noise/test_A"

# Resize all images to 256x256 before testing on them
for fname in os.listdir(results_dir):
    # print(fname)
    path = os.path.join(results_dir, fname)

    img = Image.open(path).convert("RGB").resize((256, 256))

    img.save(path)

# Apply pix2pixHD (STANDARD) on all test input images

In [7]:
# Run on output images
data_path = datadir + "data/dataset/"
results_path = datadir + "results/"
output_location = datadir + "pix2pixHD/results/day2night/test_latest/images"

!cd pix2pixHD && python test.py \
    --name day2night \
    --dataroot "{data_path}" \
    --label_nc 0 \
    --no_instance

# Copy to Google drive in results/images
!cp -r "{output_location}" \
      "{results_path}"

------------ Options -------------
aspect_ratio: 1.0
batchSize: 1
checkpoints_dir: ./checkpoints
cluster_path: features_clustered_010.npy
data_type: 32
dataroot: /content/drive/My Drive/CS 445 Final Project/data/dataset
display_winsize: 512
engine: None
export_onnx: None
feat_num: 3
fineSize: 512
fp16: False
gpu_ids: [0]
how_many: 50
input_nc: 3
instance_feat: False
isTrain: False
label_feat: False
label_nc: 0
loadSize: 1024
load_features: False
local_rank: 0
max_dataset_size: inf
model: pix2pixHD
nThreads: 2
n_blocks_global: 9
n_blocks_local: 3
n_clusters: 10
n_downsample_E: 4
n_downsample_global: 4
n_local_enhancers: 1
name: day2night
nef: 16
netG: global
ngf: 64
niter_fix_global: 0
no_flip: False
no_instance: True
norm: instance
ntest: inf
onnx: None
output_nc: 3
phase: test
resize_or_crop: scale_width
results_dir: ./results/
serial_batches: False
tf_log: False
use_dropout: False
use_encoded_image: False
verbose: False
which_epoch: latest
-------------- End ----------------
CustomDa

# Apply pix2pixHD (NOISE Adjusted) on all test input images

In [8]:
# Run on output images
data_path = datadir + "data/dataset_noise/"
results_path = datadir + "results_noise/"
output_location = datadir + "pix2pixHD/results/noisy2night/test_latest/images"

!cd pix2pixHD && python test.py \
    --name noisy2night \
    --dataroot "{data_path}" \
    --label_nc 0 \
    --no_instance

# Copy to Google drive in results_noise/images
!cp -r "{output_location}" \
      "{results_path}"

------------ Options -------------
aspect_ratio: 1.0
batchSize: 1
checkpoints_dir: ./checkpoints
cluster_path: features_clustered_010.npy
data_type: 32
dataroot: /content/drive/My Drive/CS 445 Final Project/data/dataset_noise/
display_winsize: 512
engine: None
export_onnx: None
feat_num: 3
fineSize: 512
fp16: False
gpu_ids: [0]
how_many: 50
input_nc: 3
instance_feat: False
isTrain: False
label_feat: False
label_nc: 0
loadSize: 1024
load_features: False
local_rank: 0
max_dataset_size: inf
model: pix2pixHD
nThreads: 2
n_blocks_global: 9
n_blocks_local: 3
n_clusters: 10
n_downsample_E: 4
n_downsample_global: 4
n_local_enhancers: 1
name: noisy2night
nef: 16
netG: global
ngf: 64
niter_fix_global: 0
no_flip: False
no_instance: True
norm: instance
ntest: inf
onnx: None
output_nc: 3
phase: test
resize_or_crop: scale_width
results_dir: ./results/
serial_batches: False
tf_log: False
use_dropout: False
use_encoded_image: False
verbose: False
which_epoch: latest
-------------- End ----------------

# Apply color adjustment to output images to better align with day-time images

In [ ]:
# # Color adjustment to all outputed images

# import os
# from PIL import Image
# import numpy as np
# import cv2

# results_dir = datadir + "results/images/"
# new_results = datadir + "results/adjusted/"

# # Loop through all fake_B images
# for fname in os.listdir(results_dir):
#     if fname.endswith("_synthesized_image.jpg"):
#         print(fname)
#         night_img_path = os.path.join(results_dir, fname)
#         day_img_path = os.path.join(results_dir, fname.replace("_synthesized_image.jpg", "_input_label.jpg"))

#         # Load images
#         # night_img = Image.open(night_img_path).convert("RGB")
#         # night_img = cv2.cvtColor(night_img, cv2.COLOR_RGB2Lab).astype("float32")
#         night_img = cv2.imread(night_img_path, cv2.COLOR_BGR2LAB).astype("float32")
#         # day_img = Image.open(day_img_path).convert("RGB")
#         # day_img = cv2.cvtColor(day_img, cv2.COLOR_RGB2Lab).astype("float32")
#         day_img = cv2.imread(day_img_path, cv2.COLOR_BGR2LAB).astype("float32")

#         # Color adjustment, push red-green and yellow-blue in direction of day image without changing luminance
#         result_img = np.zeros_like(night_img)
#         for i in range(256):
#             for j in range(256):
#                 result_img[i, j, 1] = (day_img[i, j, 1] + night_img[i, j, 1]) / 2
#                 result_img[i, j, 2] = (day_img[i, j, 2] + night_img[i, j, 2]) / 2

#         # Write the output image in adjusted/
#         cv2.imwrite(new_results + fname, cv2.cvtColor(result_img, cv2.COLOR_LAB2BGR))
#         # result_img.save(os.path.join(new_results, fname))

IMG_6808_synthesized_image.jpg


# Stable diffusion + Controlnet Method

(Dropped because of lots of version issues and out of memory issues)

In [ ]:
# !pip install xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.

In [ ]:
# # StableDiffusion + ControlNet method (lots of version issues and out of memory issues)

# import torch
# import cv2
# import numpy as np
# from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
# from diffusers.utils import load_image

# from PIL import Image

# controlnet = ControlNetModel.from_pretrained(
#     "lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16
# )

# pipe = StableDiffusionControlNetPipeline.from_pretrained(
#     "runwayml/stable-diffusion-v1-5", controlnet=controlnet,
#     safety_checker=None, torch_dtype=torch.float16
# )

# pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
# pipe.enable_xformers_memory_efficient_attention()
# pipe.to("cuda")
# # pipe.to("cpu")

# input_image = load_image(datadir + "data/day_image_test.jpeg")
# img = np.array(input_image)

# low_thresh, high_thresh = 100, 200
# edges = cv2.Canny(img, low_thresh, high_thresh)
# control_image = Image.fromarray(edges).convert("RGB")

# prompt = "night-time"
# negative_prompt = "sunlight, daytime, bright sky"

# output = pipe(prompt, num_inference_steps=5,
#               image=control_image,
#               negative_prompt=negative_prompt)

# output.images[0].save("output_night_image.png")


RuntimeError: Failed to import diffusers.pipelines.controlnet.pipeline_controlnet because of the following error (look up to see its traceback):
Failed to import transformers.models.clip.image_processing_clip because of the following error (look up to see its traceback):
operator torchvision::nms does not exist